# Statistical Distributions
### An Earthy Academic Reference

Classical probability distributions rendered in the project palette — **terracotta**, **ochre** and **sage** on parchment.  
Each figure overlaps three parameterisations to show how shape and location shift with parameters.

| | Continuous | Discrete |
|---|---|---|
| **Symmetric** | Normal, Student's *t* | Binomial (p=0.5) |
| **Right-skewed** | Chi-squared, Exponential, Gamma, Log-normal | Poisson |
| **Flexible** | Beta, Uniform | Binomial |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from scipy import stats


In [ ]:
# ── Palette (from site CSS) ──────────────────────────────────────────────────
BG    = '#F5F0E8'  # Parchment
UMBER = '#3D2B1F'  # Outline / text
TERRA = '#B8541C'  # Terracotta
OCHRE = '#C9971A'  # Ochre
SAGE  = '#5E7A6A'  # Sage
C3    = [TERRA, OCHRE, SAGE]

plt.rcParams.update({
    'figure.facecolor': BG,
    'axes.facecolor':   BG,
    'font.family':      'monospace',
    'text.color':       UMBER,
    'axes.labelcolor':  UMBER,
    'xtick.color':      UMBER,
    'xtick.labelsize':  9,
    'figure.dpi':       130,
})


# ── Shared helpers ───────────────────────────────────────────────────────────
def _clean(ax):
    """Strip chart junk; keep only x tick marks."""
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_yticks([])
    ax.tick_params(axis='x', colors=UMBER, length=4, width=1.2)


def mountain(ax, x, y, color, alpha=0.72):
    """Fill-under curve with thick umber outline — the logo style."""
    ax.fill_between(x, y, color=color, alpha=alpha, zorder=2)
    ln, = ax.plot(x, y, color=color, lw=2.2, zorder=3)
    ln.set_path_effects([
        pe.withStroke(linewidth=5.5, foreground=UMBER),
        pe.Normal(),
    ])
    ax.axhline(0, color=UMBER, lw=1.5, zorder=1)


def mountain_step(ax, x, y, color, alpha=0.72):
    """Same style for discrete PMFs via step rendering."""
    ax.fill_between(x, y, step='mid', color=color, alpha=alpha, zorder=2)
    ln, = ax.step(x, y, color=color, lw=2.2, where='mid', zorder=3)
    ln.set_path_effects([
        pe.withStroke(linewidth=5.5, foreground=UMBER),
        pe.Normal(),
    ])
    ax.axhline(0, color=UMBER, lw=1.5, zorder=1)


def new_ax(title, figsize=(8, 3.8)):
    fig, ax = plt.subplots(figsize=figsize, facecolor=BG)
    ax.set_facecolor(BG)
    ax.set_title(title, color=UMBER, fontsize=13, fontweight='bold',
                 loc='left', pad=10)
    _clean(ax)
    return fig, ax


def add_legend(ax, labels, colors):
    handles = [plt.Line2D([0], [0], color=c, lw=4) for c in colors]
    ax.legend(handles, labels, frameon=False, labelcolor=UMBER,
              fontsize=8.5, loc='upper right', handlelength=1.5)


---
## Continuous Distributions


### Normal (Gaussian)

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right), \quad x \in \mathbb{R}$$

| Parameter | Symbol | Role |
|---|---|---|
| Mean | **μ** | shifts the peak left/right |
| Std dev | **σ > 0** | controls width |

The symmetric bell curve. The Central Limit Theorem makes it appear everywhere sums of many independent random variables occur.


In [ ]:
fig, ax = new_ax('Normal  —  N(\u03bc, \u03c3\u00b2)')
x = np.linspace(-5.5, 5.5, 600)

curves = [
    ( 0.0, 2.0, SAGE,  '\u03bc=0,    \u03c3=2.0  (wide)'),
    (-0.8, 1.2, TERRA, '\u03bc=\u22120.8, \u03c3=1.2'),
    ( 0.8, 0.6, OCHRE, '\u03bc=0.8,  \u03c3=0.6  (narrow)'),
]
for mu, sig, col, lbl in curves:
    mountain(ax, x, stats.norm.pdf(x, mu, sig), col)

add_legend(ax, [c[3] for c in curves], [c[2] for c in curves])
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8), facecolor=BG)
ax.set_facecolor(BG)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_yticks([])
ax.set_xticks([])

x = np.linspace(-5.5, 5.5, 600)
for mu, sig, col in [(0.0, 2.0, SAGE), (-0.8, 1.2, TERRA), (0.8, 0.6, OCHRE)]:
    mountain(ax, x, stats.norm.pdf(x, mu, sig), col)

plt.tight_layout()
plt.show()


### Student's *t*

$$f(x) = \frac{\Gamma\!\left(\frac{\nu+1}{2}\right)}{\sqrt{\nu\pi}\,\Gamma\!\left(\frac{\nu}{2}\right)} \left(1+\frac{x^2}{\nu}\right)^{-\frac{\nu+1}{2}}, \quad x \in \mathbb{R}$$

| Parameter | Symbol | Role |
|---|---|---|
| Degrees of freedom | **\u03bd > 0** | \u03bd\u21920 heavier tails; \u03bd\u2192\u221e converges to N(0,1) |

At \u03bd=1 this is the Cauchy distribution (undefined mean/variance). Used in t-tests when population variance is unknown.


In [ ]:
fig, ax = new_ax("Student's t  \u2014  t(\u03bd)")
x = np.linspace(-6, 6, 600)

curves = [
    (30, SAGE,  '\u03bd=30  (\u2248 Normal)'),
    ( 5, OCHRE, '\u03bd=5'),
    ( 1, TERRA, '\u03bd=1  (Cauchy)'),
]
for nu, col, lbl in curves:
    mountain(ax, x, stats.t.pdf(x, nu), col)

add_legend(ax, [c[2] for c in curves], [c[1] for c in curves])
plt.tight_layout()
plt.show()


### Chi-squared

$$f(x) = \frac{x^{k/2-1}\,e^{-x/2}}{2^{k/2}\,\Gamma(k/2)}, \quad x > 0$$

| Parameter | Symbol | Role |
|---|---|---|
| Degrees of freedom | **k \u2265 1** | peak shifts right; shape becomes more symmetric |

Sum of **k** squared standard normal variables. Foundation of goodness-of-fit tests, the F-distribution, and likelihood-ratio tests.


In [ ]:
fig, ax = new_ax('Chi-squared  \u2014  \u03c7\u00b2(k)')
x = np.linspace(0.05, 25, 600)

curves = [
    ( 2, TERRA, 'k=2'),
    ( 5, OCHRE, 'k=5'),
    (10, SAGE,  'k=10'),
]
for k, col, lbl in curves:
    mountain(ax, x, stats.chi2.pdf(x, k), col)

add_legend(ax, [c[2] for c in curves], [c[1] for c in curves])
plt.tight_layout()
plt.show()


### Exponential

$$f(x) = \lambda\, e^{-\lambda x}, \quad x \geq 0$$

| Parameter | Symbol | Role |
|---|---|---|
| Rate | **\u03bb > 0** | mean = 1/\u03bb; higher \u03bb → steeper decay |

The **only continuous memoryless distribution**. Models waiting times between Poisson-process events (e.g., time between arrivals). Special case of Gamma with \u03b1=1.


In [ ]:
fig, ax = new_ax('Exponential  \u2014  Exp(\u03bb)')
x = np.linspace(0, 6, 600)

# scipy uses scale = 1/\u03bb
curves = [
    (0.5, SAGE,  '\u03bb=0.5  (slow decay)'),
    (1.0, OCHRE, '\u03bb=1.0'),
    (2.0, TERRA, '\u03bb=2.0  (fast decay)'),
]
for lam, col, lbl in curves:
    mountain(ax, x, stats.expon.pdf(x, scale=1/lam), col)

add_legend(ax, [c[2] for c in curves], [c[1] for c in curves])
plt.tight_layout()
plt.show()


### Gamma

$$f(x) = \frac{x^{\alpha-1}\,e^{-x/\theta}}{\theta^\alpha\,\Gamma(\alpha)}, \quad x > 0$$

| Parameter | Symbol | Role |
|---|---|---|
| Shape | **\u03b1 > 0** | \u03b1=1 \u2192 Exponential; larger \u03b1 \u2192 bell-like |
| Scale | **\u03b8 > 0** | stretches the distribution |

Describes the waiting time until the **\u03b1-th** Poisson event. Conjugate prior for the Poisson rate in Bayesian inference.


In [ ]:
fig, ax = new_ax('Gamma  \u2014  \u0393(\u03b1, \u03b8)')
x = np.linspace(0.01, 22, 600)

# scipy: gamma(a=\u03b1, scale=\u03b8)
curves = [
    (1, TERRA, '\u03b1=1  (Exponential)'),
    (3, OCHRE, '\u03b1=3'),
    (7, SAGE,  '\u03b1=7'),
]
for alpha, col, lbl in curves:
    mountain(ax, x, stats.gamma.pdf(x, a=alpha, scale=1), col)

add_legend(ax, [c[2] for c in curves], [c[1] for c in curves])
plt.tight_layout()
plt.show()


### Beta

$$f(x) = \frac{x^{\alpha-1}(1-x)^{\beta-1}}{B(\alpha,\beta)}, \quad x \in [0,1]$$

| Parameter | Role |
|---|---|
| **\u03b1 > 0** | weight on the right side; \u03b1 > \u03b2 shifts mass toward 1 |
| **\u03b2 > 0** | weight on the left side; \u03b1=\u03b2 \u2192 symmetric |

The most flexible distribution on **[0, 1]**. Natural Bayesian prior for a probability. Special cases: Uniform (\u03b1=\u03b2=1), arcsine (\u03b1=\u03b2=0.5).


In [ ]:
fig, ax = new_ax('Beta  \u2014  Beta(\u03b1, \u03b2)')
x = np.linspace(0.01, 0.99, 600)

curves = [
    (2, 5, TERRA, '\u03b1=2, \u03b2=5  (left-skewed)'),
    (2, 2, SAGE,  '\u03b1=2, \u03b2=2  (symmetric)'),
    (5, 2, OCHRE, '\u03b1=5, \u03b2=2  (right-skewed)'),
]
for a, b, col, lbl in curves:
    mountain(ax, x, stats.beta.pdf(x, a, b), col)

add_legend(ax, [c[3] for c in curves], [c[2] for c in curves])
plt.tight_layout()
plt.show()


### Log-normal

$$f(x) = \frac{1}{x\,\sigma\sqrt{2\pi}} \exp\!\left(-\frac{(\ln x - \mu)^2}{2\sigma^2}\right), \quad x > 0$$

| Parameter | Symbol | Role |
|---|---|---|
| Log-mean | **\u03bc** | controls the mode: mode = e^(\u03bc \u2212 \u03c3\u00b2) |
| Log-std | **\u03c3 > 0** | larger \u03c3 \u2192 heavier right tail |

If ln(X) ~ N(\u03bc, \u03c3\u00b2) then X is log-normal. Models multiplicative processes: incomes, stock prices, city sizes, reaction times.


In [ ]:
fig, ax = new_ax('Log-normal  \u2014  LogNorm(\u03bc, \u03c3\u00b2)')
x = np.linspace(0.01, 9, 600)

# scipy lognorm(s=\u03c3, scale=exp(\u03bc))
curves = [
    (0.0, 1.0, TERRA, '\u03bc=0,   \u03c3=1.0  (heavy tail)'),
    (0.5, 0.5, OCHRE, '\u03bc=0.5, \u03c3=0.5'),
    (1.0, 0.3, SAGE,  '\u03bc=1.0, \u03c3=0.3  (narrow)'),
]
for mu, sig, col, lbl in curves:
    mountain(ax, x, stats.lognorm.pdf(x, s=sig, scale=np.exp(mu)), col)

add_legend(ax, [c[3] for c in curves], [c[2] for c in curves])
plt.tight_layout()
plt.show()


### Uniform

$$f(x) = \frac{1}{b-a}, \quad x \in [a,\, b]$$

| Parameter | Role |
|---|---|
| **a** | lower bound |
| **b > a** | upper bound |

Maximum-entropy distribution on a bounded interval \u2014 the **\u201cI know nothing\u201d prior**. Equivalent to Beta(1,1).


In [ ]:
fig, ax = new_ax('Uniform  \u2014  U(a, b)')

intervals = [
    (0.0, 3.0, SAGE,  'a=0,   b=3   (widest)'),
    (0.5, 2.5, OCHRE, 'a=0.5, b=2.5'),
    (1.0, 2.0, TERRA, 'a=1,   b=2   (narrowest)'),
]
for a, b, col, lbl in intervals:
    h = 1 / (b - a)
    xr = np.array([a - 0.05, a,  a,  b,  b,  b + 0.05])
    yr = np.array([0,         0,  h,  h,  0,  0        ])
    mountain(ax, xr, yr, col)

add_legend(ax, [i[3] for i in intervals], [i[2] for i in intervals])
plt.tight_layout()
plt.show()


---
## Discrete Distributions


### Binomial

$$P(X=k) = \binom{n}{k}\, p^k (1-p)^{n-k}, \quad k = 0, 1, \ldots, n$$

| Parameter | Symbol | Role |
|---|---|---|
| Trials | **n \u2265 1** | total number of Bernoulli trials |
| Success prob. | **p \u2208 [0,1]** | p<0.5 \u2192 right skew; p=0.5 \u2192 symmetric; p>0.5 \u2192 left skew |

Counts successes in **n** independent Bernoulli trials. As n\u2192\u221e, p\u21920 with np=\u03bb fixed \u2192 Poisson(\u03bb).


In [ ]:
fig, ax = new_ax('Binomial  \u2014  Bin(n, p)')
x = np.arange(0, 21)

curves = [
    (20, 0.2, TERRA, 'n=20, p=0.2  (right-skewed)'),
    (20, 0.5, SAGE,  'n=20, p=0.5  (symmetric)'),
    (20, 0.8, OCHRE, 'n=20, p=0.8  (left-skewed)'),
]
for n, p, col, lbl in curves:
    mountain_step(ax, x, stats.binom.pmf(x, n, p), col)

add_legend(ax, [c[3] for c in curves], [c[2] for c in curves])
plt.tight_layout()
plt.show()


### Poisson

$$P(X=k) = \frac{\lambda^k\, e^{-\lambda}}{k!}, \quad k = 0, 1, 2, \ldots$$

| Parameter | Symbol | Role |
|---|---|---|
| Rate | **\u03bb > 0** | mean = variance = \u03bb; higher \u03bb shifts peak right |

Counts **rare events** in a fixed time/space interval. Limiting case of Binomial as n\u2192\u221e, p\u21920. Used for web hits, mutations, goal scoring.


In [ ]:
fig, ax = new_ax('Poisson  \u2014  Pois(\u03bb)')
x = np.arange(0, 28)

curves = [
    ( 2, TERRA, '\u03bb=2   (rare events)'),
    ( 7, OCHRE, '\u03bb=7'),
    (14, SAGE,  '\u03bb=14  (frequent events)'),
]
for lam, col, lbl in curves:
    mountain_step(ax, x, stats.poisson.pmf(x, lam), col)

add_legend(ax, [c[2] for c in curves], [c[1] for c in curves])
plt.tight_layout()
plt.show()


---
## Gallery

All ten distributions side by side at a glance.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 7), facecolor=BG)
fig.suptitle('Distribution Gallery', color=UMBER, fontsize=16,
             fontweight='bold', y=1.01)

def _g(ax, title):
    ax.set_facecolor(BG)
    ax.set_title(title, color=UMBER, fontsize=10, fontweight='bold')
    _clean(ax)

# Normal
ax = axes[0, 0]; x = np.linspace(-5.5, 5.5, 400)
for m, s, c in [( 0, 2.0, SAGE), (-0.8, 1.2, TERRA), (0.8, 0.6, OCHRE)]:
    mountain(ax, x, stats.norm.pdf(x, m, s), c)
_g(ax, 'Normal')

# Student's t
ax = axes[0, 1]; x = np.linspace(-6, 6, 400)
for nu, c in [(30, SAGE), (5, OCHRE), (1, TERRA)]:
    mountain(ax, x, stats.t.pdf(x, nu), c)
_g(ax, "Student's t")

# Chi-squared
ax = axes[0, 2]; x = np.linspace(0.05, 25, 400)
for k, c in [(2, TERRA), (5, OCHRE), (10, SAGE)]:
    mountain(ax, x, stats.chi2.pdf(x, k), c)
_g(ax, 'Chi-squared')

# Exponential
ax = axes[0, 3]; x = np.linspace(0, 6, 400)
for lam, c in [(0.5, SAGE), (1.0, OCHRE), (2.0, TERRA)]:
    mountain(ax, x, stats.expon.pdf(x, scale=1/lam), c)
_g(ax, 'Exponential')

# Gamma
ax = axes[0, 4]; x = np.linspace(0.01, 22, 400)
for a, c in [(1, TERRA), (3, OCHRE), (7, SAGE)]:
    mountain(ax, x, stats.gamma.pdf(x, a=a, scale=1), c)
_g(ax, 'Gamma')

# Beta
ax = axes[1, 0]; x = np.linspace(0.01, 0.99, 400)
for a, b, c in [(2, 5, TERRA), (2, 2, SAGE), (5, 2, OCHRE)]:
    mountain(ax, x, stats.beta.pdf(x, a, b), c)
_g(ax, 'Beta')

# Log-normal
ax = axes[1, 1]; x = np.linspace(0.01, 9, 400)
for mu, sig, c in [(0, 1.0, TERRA), (0.5, 0.5, OCHRE), (1.0, 0.3, SAGE)]:
    mountain(ax, x, stats.lognorm.pdf(x, s=sig, scale=np.exp(mu)), c)
_g(ax, 'Log-normal')

# Uniform
ax = axes[1, 2]
for a, b, c in [(0, 3, SAGE), (0.5, 2.5, OCHRE), (1, 2, TERRA)]:
    h = 1 / (b - a)
    mountain(ax, np.array([a-.05,a,a,b,b,b+.05]), np.array([0,0,h,h,0,0]), c)
_g(ax, 'Uniform')

# Binomial
ax = axes[1, 3]; x = np.arange(0, 21)
for p, c in [(0.2, TERRA), (0.5, SAGE), (0.8, OCHRE)]:
    mountain_step(ax, x, stats.binom.pmf(x, 20, p), c)
_g(ax, 'Binomial')

# Poisson
ax = axes[1, 4]; x = np.arange(0, 28)
for lam, c in [(2, TERRA), (7, OCHRE), (14, SAGE)]:
    mountain_step(ax, x, stats.poisson.pmf(x, lam), c)
_g(ax, 'Poisson')

plt.tight_layout()
plt.show()
